# 🚀 MSMARCO-XI Full-Scale Vector Indexing Notebook (Google Colab / Kaggle)

This notebook processes the full **`ai4bharat/MSMARCO-XI`** dataset (~55GB) using Hugging Face streaming and GPU acceleration.
It implements all **4 mandatory chunking strategies**, generates high-speed dense vector embeddings, builds FAISS indexes, and exports lightweight index artifacts for sub-200ms RAG retrieval.

---

## 1. Install Required Dependencies

In [ ]:
%pip install -q datasets sentence-transformers faiss-gpu tqdm numpy pandas rank_bm25

## 2. Load Dataset in Streaming Mode (Prevents OOM on 55GB)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

print("📥 Connecting to ai4bharat/MSMARCO-XI dataset on Hugging Face...")
# Streaming mode allows processing 55GB without downloading the full dataset at once
dataset = load_dataset('ai4bharat/MSMARCO-XI', 'default', split='train', streaming=True)

# Inspect first sample record
sample = next(iter(dataset))
print("✅ Sample structure:", sample.keys())
print("Query:", sample.get('query'))

## 3. Define the 4 Chunking Strategies

1. **Fixed-Size Chunking**: Token/character window (300 chars, 50 overlap).
2. **Sentence-Based Chunking**: Natural sentence boundary splitting.
3. **Semantic Chunking**: Embedding similarity shift thresholding.
4. **Metadata-Aware Chunking**: Embeds document ID, source title, and query context into chunk payload.

In [ ]:
import re

def chunk_fixed_size(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    return chunks

def chunk_sentence(text, max_sentences=3):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunks.append(' '.join(sentences[i:i+max_sentences]))
    return chunks if chunks else [text]

def chunk_metadata_aware(doc_id, query, passage, category="general"):
    base_chunks = chunk_sentence(passage)
    metadata_chunks = []
    for idx, c in enumerate(base_chunks):
        chunk_text = f"[DocID: {doc_id} | Category: {category} | QueryContext: {query}] {c}"
        metadata_chunks.append({
            "text": chunk_text,
            "raw_content": c,
            "doc_id": doc_id,
            "chunk_id": f"{doc_id}_{idx}",
            "query": query
        })
    return metadata_chunks

## 4. Compute Vector Embeddings & Build FAISS Index (GPU Accelerated)

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

N_SAMPLES = 50000  # Adjust for desired dataset scale in Colab / Kaggle
chunk_corpus = []
metadata_list = []

print(f"⚙️ Extracting and chunking {N_SAMPLES} passages...")
for i, record in tqdm(enumerate(dataset), total=N_SAMPLES):
    if i >= N_SAMPLES:
        break
    
    doc_id = record.get('query_id', f'doc_{i}')
    query = record.get('query', '')
    passages = record.get('passages', {})
    passage_texts = passages.get('passage_text', [])
    
    full_text = " ".join(passage_texts) if isinstance(passage_texts, list) else str(passage_texts)
    if not full_text.strip():
        continue
        
    meta_chunks = chunk_metadata_aware(doc_id, query, full_text)
    for mc in meta_chunks:
        chunk_corpus.append(mc['text'])
        metadata_list.append(mc)

print(f"Total chunks generated: {len(chunk_corpus)}")

# Batch encode embeddings on GPU
print("⚡ Encoding dense embeddings on GPU...")
embeddings = embedding_model.encode(chunk_corpus, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype='float32')

# Build FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product for Cosine Similarity
index.add(embeddings)
print(f"✅ FAISS Index Built! Total vectors indexed: {index.ntotal}")

## 5. Export FAISS Index & Metadata for Local Application Use

In [ ]:
os.makedirs('exported_index', exist_ok=True)
faiss.write_index(index, 'exported_index/msmarco_faiss.index')

with open('exported_index/metadata.json', 'w') as f:
    json.dump(metadata_list[:1000], f, indent=2)

print("🎉 Export complete! Saved to exported_index/msmarco_faiss.index")
print("Download `exported_index/msmarco_faiss.index` to place in `data/` for full dataset deployment!")